In [ ]:
%matplotlib widget

In [ ]:
from glob import glob
import numpy as np
import pandas as pd
import flammkuchen as fl
from split_dataset import SplitDataset
from bouter import Experiment
from fimpy.pipeline.general import calc_f0, dff
from motions.utilities import stim_vel_dir_dataframe, quantize_directions
from scipy.interpolate import interp1d 
from scipy.signal import convolve2d
import colorspacious
import napari
import matplotlib.pyplot as plt

from fimpylab.core.twop_experiment import TwoPExperiment

from pathlib import Path

In [ ]:
# find the frames to calculate the baseline.
def no_regressor_frames(regressors, threshold=0.01):
    return np.where(np.all(regressors.values < threshold, axis=1))[0]

# calculate the baseline, plane-wise
def calc_f0(stack, frames):
    fr_mean = None
    for i_frame in frames:
        sf = stack[int(i_frame), :]
        if fr_mean is None:
            fr_mean = sf
        else:
            fr_mean += sf
    return fr_mean / len(frames)

In [ ]:
# calculate dot product with each regressor from dF/F traces, px-wise
def get_tuning_map(img, sens_regs, n_dirs=8):
    traces = img.reshape(img.shape[0], -1)

    n_t = sens_regs.shape[1]
    print(np.shape(img))
    print(np.shape(traces))
    reg = sens_regs.T @ traces[:n_t, :]
    reg = reg.reshape(reg.shape[0], img.shape[-1], img.shape[-1])

    return reg

In [ ]:
master = Path(r"Z:\Hagar\e0075\ablation\control")
fish_list = list(master.glob("*_f*"))
fish = fish_list[0]
print(fish)

aligned = SplitDataset(fish / "aligned")
exp_list = glob(str(fish / "behavior/*.json"))

sampling = 1/3
time = np.linspace(0, aligned.shape[0]*sampling, aligned.shape[0])

In [ ]:
len_rec, num_planes, x_pix, y_pix = np.shape(aligned)
np.shape(aligned)

In [ ]:
fish = fish_list[5]
print(fish)

In [ ]:
corrmap_all = fl.load(fish / "plane_corrmap_corrvalues.h5")['plane_corr']

In [ ]:
title_list = ['■□□□□□□□', '□■□□□□□□', '□□■□□□□□', '□□□■□□□□', '□□□□■□□□', '□□□□□■□□', '□□□□□□■□', '□□□□□□□■']

In [ ]:
np.shape(corrmap_all)

In [ ]:
ind_plane = 4
corrmap = corrmap_all[ind_plane]

In [ ]:
num_row = 2
num_col = 4
fig, axs = plt.subplots(num_row, num_col, figsize=(10, 5), sharey=True, sharex=True)

count = 0 
vmax = 0.25
fig.suptitle("Plane " + str (ind_plane) + ', corr thresh=' + str(vmax))

for i in range(0, num_row*num_col):
    r = i // num_col
    c = np.mod(i, num_col)
    
    if count > 0:
        axs[r,c].axis('off')
    else:
        count += 1
    
    tmp_plane = np.rot90(corrmap[i], 3)
    #tmp_plane = np.ma.masked_where(tmp_anatomy < 1, tmp_plane)
    axs[r,c].imshow(tmp_plane, cmap='coolwarm', vmin=-vmax, vmax=vmax)
    axs[r,c].set_title(title_list[i])

In [ ]:
#fig.savefig(fish / "plane0_regs_map.pdf", dpi=300)
file_name = "plane" + str(ind_plane) + "_regs_map_corrval_025.jpg"
fig.savefig(fish / file_name, dpi=300)
file_name = "plane" + str(ind_plane) + "_regs_map_corrval_025.pdf"
fig.savefig(fish / file_name, dpi=300)